# OmniSight: Kaggle GPU inference node

Serves **Qwen2-VL-7B-Instruct** (4-bit NF4) plus **Whisper** behind a Cloudflare quick tunnel, and publishes the live URL to your GitHub Gist.

**Before running**
1. **Settings → Accelerator:** `GPU T4 x2` (the node uses one GPU) or `GPU P100`.
2. **Settings → Internet:** on.
3. **Add-ons → Secrets:** attach `GITHUB_TOKEN` (gist scope only). Optional: `OMNISIGHT_GIST_ID` (defaults to the project gist), `OMNISIGHT_API_KEY`, `HF_TOKEN`.
4. `REPO_URL` in the next cell points at the project repository; change it if you fork.

The node runs in a **fresh Python process** (`launch.py`), so the pinned torch build is used without restarting the kernel. For long sessions, prefer **Save Version → Save & Run All** over an interactive session. Also check Kaggle's current terms on public endpoints and idle loops.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Lin-2352/omnisight.git"  # change if you fork the project
REPO_DIR = "/kaggle/working/omnisight"
# Public gist that receives the live endpoint (not a secret; an attached
# OMNISIGHT_GIST_ID secret takes precedence).
DEFAULT_GIST_ID = "fc8433475a1a2d424ad8e06724e59731"
RUN_BENCHMARK = False  # True: benchmark in-process first (the model cannot be loaded twice on one GPU)

assert "<" not in REPO_URL, "Edit REPO_URL to point at your OmniSight repository"

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Pinned stack (torch 2.3.1, transformers <4.50, bitsandbytes <0.46) + shared contracts.
!{sys.executable} -m pip install -q -r {REPO_DIR}/kaggle-server/requirements-kaggle.txt
!{sys.executable} -m pip install -q --no-deps {REPO_DIR}

In [ ]:
# Export Kaggle Secrets into this kernel's environment; child processes inherit them.
# Values are never printed.
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
for _name in ("GITHUB_TOKEN", "OMNISIGHT_GIST_ID", "OMNISIGHT_API_KEY", "HF_TOKEN"):
    try:
        _value = _secrets.get_secret(_name)
    except Exception:
        _value = None
    if _value:
        os.environ[_name] = _value.strip()
    print(f"{_name:<20} {'attached' if _value else 'not attached'}")
del _secrets
if not os.environ.get("OMNISIGHT_GIST_ID") and DEFAULT_GIST_ID:
    os.environ["OMNISIGHT_GIST_ID"] = DEFAULT_GIST_ID
    print(f"{'OMNISIGHT_GIST_ID':<20} using DEFAULT_GIST_ID")


In [ ]:
# Optional: synthetic TTFT / tokens-per-second / VRAM benchmark (report saved under kaggle-server/benchmark-results/).
if RUN_BENCHMARK:
    !cd {REPO_DIR} && {sys.executable} -u kaggle-server/benchmark.py --mode inprocess --runs 3

In [ ]:
# Start the node. The cell keeps running while the node serves; interrupt it to publish "offline" and stop.
!cd {REPO_DIR} && {sys.executable} -u kaggle-server/launch.py